# KQC7016 Assignment 1 - Smart Hospital: Question-based EDA Notebook

This notebook supports the **question-based data story report**.

The theme stays the same: **Smart Hospital - Heart Risk Screening from Routine Check-up Data**.

The notebook is written in simple steps so a non-medical reader can follow the work:

1. Load and clean the data.
2. Explore the data with charts.
3. Test the main patterns statistically.
4. Save figures and tables for the report.

Dataset source: UCI Machine Learning Repository - Heart Disease Dataset  
Dataset DOI: https://doi.org/10.24432/C52P4X

**Important note:** This notebook is for data analytics practice. It is not a medical diagnosis tool.


## Step 1 - Set up Python packages

This cell imports the packages used in the notebook. It also creates folders for saved figures and cleaned data.


In [ ]:
# Required packages: pandas, numpy, matplotlib, scipy, statsmodels

import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from scipy import stats
from statsmodels.stats.multitest import multipletests

os.makedirs("outputs", exist_ok=True)
os.makedirs("data", exist_ok=True)

# Make charts easier to read in the report.
plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 160,
    "savefig.dpi": 300,
})

def add_value_labels(ax, fmt="{:.0f}", ypad=0.02):
    """Add number labels above bar charts."""
    ymin, ymax = ax.get_ylim()
    offset = (ymax - ymin) * ypad
    for patch in ax.patches:
        h = patch.get_height()
        ax.text(
            patch.get_x() + patch.get_width() / 2,
            h + offset,
            fmt.format(h),
            ha="center",
            va="bottom",
            fontsize=10,
        )


## Step 2 - Load the dataset

This cell loads the Cleveland data from UCI. If the online link fails, download `processed.cleveland.data` manually and place it either in the same folder as this notebook or inside the `data` folder.


In [ ]:
# Load data from a local file first, then try the UCI online source if needed.
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"
]

local_paths = [
    "data/processed.cleveland.data",
    "processed.cleveland.data",
]

df_raw = None

# Local-first loading makes the notebook more reproducible when it is submitted with the data file.
for path in local_paths:
    if os.path.exists(path):
        df_raw = pd.read_csv(path, header=None, names=columns, na_values="?")
        print(f"Loaded local file: {path}")
        break

# If no local file is found, try the online UCI source.
if df_raw is None:
    try:
        df_raw = pd.read_csv(url, header=None, names=columns, na_values="?")
        print("Loaded data from UCI online source.")
    except Exception as e:
        raise FileNotFoundError(
            "Data loading failed. Please download processed.cleveland.data and place it in the notebook folder or the data folder."
        ) from e

df_raw.head()


## Step 3 - Understand the raw data

This cell checks the number of rows and columns, summary statistics, missing values, duplicate rows, and target values. This gives us a quick picture before cleaning.


In [ ]:
# Basic data understanding
print("Shape:", df_raw.shape)
display(df_raw.info())
display(df_raw.describe(include="all"))

missing = df_raw.isna().sum()
print("Missing values by column:")
display(missing[missing > 0])

print("Duplicate rows:", df_raw.duplicated().sum())
print("Target values:")
display(df_raw["target"].value_counts().sort_index())


## Step 4 - Clean the data

Only two columns have missing values: `ca` and `thal`. Because the number of missing cells is very small, we fill them with the most common value in each column. Then we create a simple two-group outcome: `No disease` and `Disease present`.


In [ ]:
# Data cleaning

df = df_raw.copy()

# Fill missing values in ca and thal using the most common value.
for col in ["ca", "thal"]:
    df[col] = df[col].fillna(df[col].mode()[0])

# Convert category-style numeric columns to integer labels.
for col in ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal", "target"]:
    df[col] = df[col].astype(int)

# Binary target for the main comparison: 0 = no disease, 1 = disease present.
df["disease"] = (df["target"] > 0).astype(int)
df["disease_label"] = df["disease"].map({0: "No disease", 1: "Disease present"})

# Plain-language labels for selected category variables.
cp_map = {1: "Typical chest pain", 2: "Atypical chest pain", 3: "Other chest pain", 4: "No clear symptoms"}
exang_map = {0: "No", 1: "Yes"}
thal_map = {3: "Normal", 6: "Fixed issue", 7: "Reversible issue"}

df["cp_label"] = df["cp"].map(cp_map)
df["exang_label"] = df["exang"].map(exang_map)
df["thal_label"] = df["thal"].map(thal_map)

df.to_csv("data/heart_cleveland_cleaned.csv", index=False)
print("Cleaned dataset saved to data/heart_cleveland_cleaned.csv")
df.head()


## Exploratory Data Analysis (EDA): question-based view

EDA means looking at the data before formal testing. In this notebook, each chart answers a plain question:

- Is the dataset balanced enough to compare groups?
- Which numeric measurements look different between groups?
- Which category-style fields give clearer warning signs?
- Which variables move together with recorded disease severity?


### Question A - Is the dataset balanced enough to compare groups?

This figure checks two things: group size and missing data. If one group is too small, comparison is weaker. If there are many missing values, the cleaning decision becomes more important.


In [ ]:
# Figure 1: class balance and missing data distribution (n=303)
fig, axes = plt.subplots(1, 2, figsize=(12.8, 5.0))

counts = df["disease_label"].value_counts().reindex(["No disease", "Disease present"])
axes[0].bar(counts.index, counts.values)
axes[0].set_title("Disease-status group balance (n=303)")
axes[0].set_xlabel("Disease status")
axes[0].set_ylabel("Number of patients")
axes[0].yaxis.set_major_locator(MaxNLocator(integer=True))
add_value_labels(axes[0])

missing_before = df_raw.isna().sum()
missing_before = missing_before[missing_before > 0]
axes[1].bar(missing_before.index, missing_before.values)
axes[1].set_title("Missing data before cleaning")
axes[1].set_xlabel("Variable")
axes[1].set_ylabel("Missing cells")
axes[1].yaxis.set_major_locator(MaxNLocator(integer=True))
add_value_labels(axes[1])

fig.suptitle("Figure 1. Class balance and missing data distribution (n=303)", fontsize=15, y=1.02)
fig.tight_layout()
plt.savefig("outputs/figure_1_class_balance_missing_data.png", bbox_inches="tight")
plt.show()


### Question B - Which numeric measurements look different between the two groups?

This figure compares selected numeric variables between the no-disease group and the disease-present group. The boxplots show the middle range. The dots show individual records.


In [ ]:
# Figure 2: selected health measurements by disease status
plot_vars = [
    ("age", "Age", "Age (years)"),
    ("trestbps", "Resting blood pressure", "Blood pressure (mmHg)"),
    ("thalach", "Highest heart rate in exercise test", "Heart rate (bpm)"),
    ("oldpeak", "Exercise-test change (oldpeak)", "Oldpeak score"),
]

fig, axes = plt.subplots(2, 2, figsize=(11.0, 7.0))
for ax, (col, title, ylabel) in zip(axes.ravel(), plot_vars):
    group_data = [
        df.loc[df["disease"] == 0, col].dropna(),
        df.loc[df["disease"] == 1, col].dropna(),
    ]
    ax.boxplot(group_data, tick_labels=["No disease", "Disease present"], showfliers=True)
    rng = np.random.default_rng(10)
    for i, values in enumerate(group_data, start=1):
        x = rng.normal(i, 0.035, size=len(values))
        ax.scatter(x, values, alpha=0.35, s=10)
    ax.set_title(title)
    ax.set_xlabel("Disease status")
    ax.set_ylabel(ylabel)
    ax.grid(True, axis="y", alpha=0.25)

fig.suptitle("Health measurement distributions by disease status", fontsize=15, y=1.02)
fig.tight_layout()
plt.savefig("outputs/figure_2_health_measurements_by_status.png", bbox_inches="tight")
plt.show()


### Question C - Which category-style fields give clearer warning signs?

This figure shows the percentage of disease-present records inside each category. It helps us see which categories may deserve more attention in a smart hospital screening dashboard.


In [ ]:
# Figure 3: disease-present percentage by selected category indicators
cat_specs = [
    ("cp", "Chest pain type", {1: "Typical chest pain", 2: "Atypical chest pain", 3: "Other chest pain", 4: "No clear symptoms"}),
    ("exang", "Chest pain during exercise", {0: "No", 1: "Yes"}),
    ("ca", "Vessels shown in scan", {0: "0", 1: "1", 2: "2", 3: "3"}),
    ("thal", "Thal test result", {3: "Normal", 6: "Fixed issue", 7: "Reversible issue"}),
]

fig, axes = plt.subplots(2, 2, figsize=(11.4, 7.1))
for ax, (col, title, mapping) in zip(axes.ravel(), cat_specs):
    rows = []
    for value, label in mapping.items():
        subset = df[df[col] == value]
        if len(subset) > 0:
            rows.append((label, subset["disease"].mean() * 100, len(subset)))
    labels = [r[0] for r in rows]
    percentages = [r[1] for r in rows]
    bars = ax.bar(labels, percentages)
    ax.set_title(title)
    ax.set_xlabel("Category")
    ax.set_ylabel("Disease present (%)")
    ax.set_ylim(0, 100)
    ax.tick_params(axis="x", rotation=20)
    ax.grid(True, axis="y", alpha=0.25)
    for patch, pct in zip(bars, percentages):
        ax.text(patch.get_x() + patch.get_width()/2, pct + 2,
                f"{pct:.1f}%", ha="center", va="bottom", fontsize=9)

fig.suptitle("Disease-present percentage by selected category indicators", fontsize=15, y=1.02)
fig.tight_layout()
plt.savefig("outputs/figure_3_category_indicators.png", bbox_inches="tight")
plt.show()


### Question D - Which variables move together with recorded disease severity?

This heatmap uses Spearman correlation. A positive value means two variables tend to increase together. A negative value means one variable tends to decrease when the other increases.


In [ ]:
# Figure 4: Spearman correlation heatmap with simple names
corr_vars = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca", "target"]
full_names = [
    "Age",
    "Resting blood pressure",
    "Cholesterol level",
    "Highest heart rate",
    "Exercise-test change",
    "Vessels shown in scan",
    "Disease severity",
]

spear = df[corr_vars].corr(method="spearman")
fig, ax = plt.subplots(figsize=(8.8, 7.7))
im = ax.imshow(spear.values, vmin=-1, vmax=1)
ax.set_xticks(np.arange(len(full_names)))
ax.set_yticks(np.arange(len(full_names)))
ax.set_xticklabels(full_names, rotation=45, ha="right")
ax.set_yticklabels(full_names)

for i in range(len(full_names)):
    for j in range(len(full_names)):
        ax.text(j, i, f"{spear.values[i, j]:.2f}", ha="center", va="center", fontsize=10)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Spearman rho")
ax.set_title("Spearman correlation among selected variables and recorded risk level")
fig.tight_layout()
plt.savefig("outputs/figure_4_spearman_heatmap.png", bbox_inches="tight")
plt.show()

display(spear["target"].sort_values(ascending=False))


## Statistical Analysis: checking whether the patterns are reliable

The charts show possible patterns. Statistical tests help us check whether the patterns are likely to be random.

All tests use **alpha = 0.05**. A result is treated as statistically significant when `p < 0.05`.


### Helper functions

These small helper functions calculate effect sizes and format p-values. They keep the later testing cells shorter and easier to read.


In [ ]:
# Helper functions for statistical testing

def cliffs_delta(x, y):
    """Return Cliff's delta for two independent samples."""
    x = np.asarray(x)
    y = np.asarray(y)
    greater = 0
    lower = 0
    for xi in x:
        greater += np.sum(xi > y)
        lower += np.sum(xi < y)
    return (greater - lower) / (len(x) * len(y))

def cramers_v(table):
    """Return Cramer's V for a chi-square table."""
    chi2 = stats.chi2_contingency(table)[0]
    n = table.sum().sum()
    r, k = table.shape
    return np.sqrt((chi2 / n) / min(k - 1, r - 1))

def fmt_p(p):
    """Show very small p-values as <0.001."""
    return "<0.001" if p < 0.001 else f"{p:.3f}"


### Assumption checks for numeric variables

Before choosing the test, we check normality and equal variance. The results are used to justify a non-parametric test, Mann-Whitney U, for two-group comparisons.


In [ ]:
# Assumption checks for continuous variables
continuous_test_vars = ["age", "trestbps", "chol", "thalach", "oldpeak"]
assumption_rows = []

for var in continuous_test_vars:
    disease = df.loc[df["disease"] == 1, var].dropna()
    no_disease = df.loc[df["disease"] == 0, var].dropna()
    shapiro_disease = stats.shapiro(disease)
    shapiro_no_disease = stats.shapiro(no_disease)
    levene_test = stats.levene(disease, no_disease)
    assumption_rows.append({
        "variable": var,
        "shapiro_p_disease": shapiro_disease.pvalue,
        "shapiro_p_no_disease": shapiro_no_disease.pvalue,
        "levene_p": levene_test.pvalue,
        "normality_issue": (shapiro_disease.pvalue < 0.05) or (shapiro_no_disease.pvalue < 0.05),
        "variance_issue": levene_test.pvalue < 0.05,
    })

assumption_results = pd.DataFrame(assumption_rows)
display(assumption_results)


### RQ1 - Numeric measurements

The Mann-Whitney U test compares the no-disease group and the disease-present group for numeric variables. It is used because several variables do not follow a normal distribution.


In [ ]:
# Mann-Whitney U tests for continuous variables
rows = []

for var in continuous_test_vars:
    disease = df.loc[df["disease"] == 1, var]
    no_disease = df.loc[df["disease"] == 0, var]
    u_stat, p_value = stats.mannwhitneyu(disease, no_disease, alternative="two-sided")
    delta = cliffs_delta(disease, no_disease)
    rows.append({
        "variable": var,
        "disease_median": disease.median(),
        "no_disease_median": no_disease.median(),
        "u_statistic": u_stat,
        "p": p_value,
        "p_formatted": fmt_p(p_value),
        "cliffs_delta": round(delta, 2),
    })

continuous_results = pd.DataFrame(rows)
display(continuous_results)


### RQ2 - Category-style indicators

The Chi-square test checks whether a category-style variable is linked with disease status. Cramer's V is included to show whether the link is weak, medium, or strong.


In [ ]:
# Chi-square tests for category indicators
categorical_test_vars = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
rows = []

for var in categorical_test_vars:
    table = pd.crosstab(df[var], df["disease"])
    chi2, p_value, dof, expected = stats.chi2_contingency(table)
    rows.append({
        "variable": var,
        "chi2": chi2,
        "df": dof,
        "p": p_value,
        "p_formatted": fmt_p(p_value),
        "cramers_v": round(cramers_v(table), 2),
        "min_expected": round(expected.min(), 1),
    })

categorical_results = pd.DataFrame(rows)
display(categorical_results)


### RQ3 - Exercise-test variables across severity classes

The Kruskal-Wallis test compares more than two recorded disease levels. If the result is significant, post-hoc tests help show which levels are different.


In [ ]:
# Kruskal-Wallis tests for severity classes 0-4
severity_vars = ["oldpeak", "thalach"]
rows = []
for var in severity_vars:
    groups = [df.loc[df["target"] == cls, var] for cls in sorted(df["target"].unique())]
    h_stat, p_value = stats.kruskal(*groups)
    rows.append({
        "variable": var,
        "test": "Kruskal-Wallis H",
        "statistic": h_stat,
        "p": p_value,
        "p_formatted": fmt_p(p_value),
    })
severity_results = pd.DataFrame(rows)
display(severity_results)

# Pairwise post-hoc comparisons using Mann-Whitney U with Holm correction.
# This matches the report wording: it is not Dunn's test and not Bonferroni correction.
posthoc_rows = []
classes = sorted(df["target"].unique())
for var in severity_vars:
    p_values = []
    pairs = []
    for a, b in itertools.combinations(classes, 2):
        x = df.loc[df["target"] == a, var]
        y = df.loc[df["target"] == b, var]
        u_stat, p_value = stats.mannwhitneyu(x, y, alternative="two-sided")
        pairs.append((var, a, b, u_stat, p_value))
        p_values.append(p_value)
    rejected, p_adj, _, _ = multipletests(p_values, method="holm")
    for (var, a, b, u_stat, p_value), holm_p, significant in zip(pairs, p_adj, rejected):
        posthoc_rows.append({
            "variable": var,
            "class_a": a,
            "class_b": b,
            "u_statistic": u_stat,
            "raw_p": p_value,
            "raw_p_formatted": fmt_p(p_value),
            "holm_adjusted_p": holm_p,
            "holm_adjusted_p_formatted": fmt_p(holm_p),
            "significant_after_holm": significant,
        })

posthoc_results = pd.DataFrame(posthoc_rows)
display(posthoc_results)


### Figure 5 - Exercise-test readings across recorded disease levels

This figure shows how exercise-test ST change and highest exercise heart rate vary across the original 0-4 recorded disease levels.


In [ ]:
# Figure 5: exercise-test change and highest heart rate across recorded risk levels
severity_classes = sorted(df["target"].unique())
class_labels = [str(x) for x in severity_classes]
oldpeak_data = [df.loc[df["target"] == cls, "oldpeak"] for cls in severity_classes]
thalach_data = [df.loc[df["target"] == cls, "thalach"] for cls in severity_classes]

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.0))
axes[0].boxplot(oldpeak_data, tick_labels=class_labels, showfliers=True)
axes[0].set_title("Exercise-test change by risk level")
axes[0].set_xlabel("Recorded risk level (0-4)")
axes[0].set_ylabel("Exercise-test change score")
axes[0].grid(True, axis="y", alpha=0.25)

axes[1].boxplot(thalach_data, tick_labels=class_labels, showfliers=True)
axes[1].set_title("Highest heart rate by risk level")
axes[1].set_xlabel("Recorded risk level (0-4)")
axes[1].set_ylabel("Heart rate (bpm)")
axes[1].grid(True, axis="y", alpha=0.25)

fig.suptitle("Exercise-test change and highest heart rate across recorded risk levels", fontsize=15, y=1.02)
fig.tight_layout()
plt.savefig("outputs/figure_5_severity_boxplots.png", bbox_inches="tight")
plt.show()


## Main findings written in report language

- The dataset is usable for basic comparison because the two groups are fairly balanced.
- Only six cells are missing, so simple mode filling is acceptable for this student-level EDA task.
- Exercise-test variables give clearer signals than blood pressure or cholesterol alone.
- Category-style fields such as chest pain type, exercise-related chest pain, colored blood-vessel count, and stress-test result are strongly linked with disease status.
- The results can support a smart hospital screening dashboard, but they should not be used as an automatic diagnosis system.
